 Bearing RUL Modelling
 Arya Dinesh Chirakkuni
This notebook handles model training and prediction on test bearings.
It loads the features extracted in rul_prediction.ipynb

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import kurtosis, skew
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import warnings
warnings.filterwarnings("ignore")

 Step 1: Load Training Features
Load the features extracted by the feature extraction notebook.

In [ ]:
df = pd.read_csv("train_features.csv")
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Bearings: {df['bearing'].unique()}")

Step 2: Normalize RUL
Bearings have very different total lifetimes (1h to 7h).
We normalize RUL as a fraction of total life [0=fail, 1=new]
so the model learns degradation patterns, not absolute time.

In [ ]:
df["RUL_norm"] = df["RUL"] / df["total_steps"]
df["life_fraction"] = df["time_step"] / df["total_steps"]

print("RUL normalized to [0, 1]")
print(df[["bearing", "RUL", "total_steps", "RUL_norm"]].head())

Step 3: Prepare Features and Target
12 vibration features + life_fraction as input.
Normalized RUL as target.

In [ ]:
feature_cols = [
    "rms_x", "peak_x", "kurtosis_x", "skew_x", "std_x", "crest_x",
    "rms_y", "peak_y", "kurtosis_y", "skew_y", "std_y", "crest_y"
]

X = df[feature_cols].values
y = df["RUL_norm"].values
groups = df["bearing"].values

print(f"Features: {len(feature_cols)}")
print(f"Samples: {len(X)}")

Step 4: Train/Validation Split by Bearing
Split by bearing not by row to avoid data leakage.
Train on 4 bearings, validate on 2.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f"Train bearings: {set(groups[train_idx])}")
print(f"Val bearings:   {set(groups[val_idx])}")
print(f"Overlap: {set(groups[train_idx]) & set(groups[val_idx])} ← must be empty")

Step 5: Train Models and Pick Best
Compare Random Forest vs Gradient Boosting.
Random Forest gives natural uncertainty via tree variance.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import joblib

# Train on all conditions together but with better features
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_r2 = r2_score(y_val, rf.predict(X_val))
print(f"Random Forest → R²: {rf_r2:.4f}")

best_model = rf
best_name = "Random Forest"

# Retrain on ALL data
best_model.fit(X, y)
joblib.dump(best_model, "best_rul_model.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")
print("✅ Saved: best_rul_model.pkl")

Step 6: Extract Features from Actual Test Bearings
Load raw vibration files from Test_set folder and extract features.

In [ ]:
def extract_features(filepath):
    df_raw = pd.read_csv(filepath, header=None)
    ax = df_raw[4].values
    ay = df_raw[5].values if 5 in df_raw.columns else ax
    features = {}
    for name, sig in [("x", ax), ("y", ay)]:
        features[f"rms_{name}"]      = np.sqrt(np.mean(sig**2))
        features[f"peak_{name}"]     = np.max(np.abs(sig))
        features[f"kurtosis_{name}"] = kurtosis(sig)
        features[f"skew_{name}"]     = skew(sig)
        features[f"std_{name}"]      = np.std(sig)
        features[f"crest_{name}"]    = np.max(np.abs(sig)) / (np.sqrt(np.mean(sig**2)) + 1e-10)
    return features

test_path = "Test_set"
all_test = []
for bearing in sorted(os.listdir(test_path)):
    path = os.path.join(test_path, bearing)
    files = sorted([f for f in os.listdir(path) if f.startswith("acc_")])
    records = []
    for i, f in enumerate(files):
        feats = extract_features(os.path.join(path, f))
        feats["time_step"] = i
        feats["total_steps"] = len(files)
        feats["bearing"] = bearing
        records.append(feats)
    all_test.append(pd.DataFrame(records))
    print(f"  {bearing}: {len(files)} files")

test_df = pd.concat(all_test, ignore_index=True)
test_df["life_fraction"] = test_df["time_step"] / test_df["total_steps"]
print(f"\nTotal test rows: {test_df.shape[0]}")

Step 7: Predict RUL with Per-Bearing Uncertainty
Use tree variance for uncertainty — each tree gives a slightly
different prediction, the spread = how confident the model is.

In [ ]:
X_test = test_df[feature_cols].values

preds_trees = np.array([tree.predict(X_test) for tree in best_model.estimators_])
y_pred_norm = preds_trees.mean(axis=0)
y_std_norm  = preds_trees.std(axis=0)

# Use LAST 50 steps smoothed prediction per bearing
test_df["pred_norm"] = np.clip(y_pred_norm, 0, 1)
test_df["std_norm"]  = y_std_norm

# Smooth predictions with rolling average to reduce noise
smoothed = []
stds = []
for bearing in test_df["bearing"].unique():
    mask = test_df["bearing"] == bearing
    preds = test_df.loc[mask, "pred_norm"]
    stds_b = test_df.loc[mask, "std_norm"]
    smoothed_preds = preds.rolling(20, min_periods=1).mean()
    smoothed.extend(smoothed_preds.values)
    stds.extend(stds_b.values)

test_df["pred_smooth"] = smoothed
test_df["std_smooth"]  = stds

# Convert to seconds using actual total steps per bearing
bearing_total = test_df.groupby("bearing")["total_steps"].first()
test_df["Predicted_RUL_s"] = test_df["pred_smooth"] * \
    test_df["bearing"].map(bearing_total) * 10
test_df["Uncertainty_s"]   = test_df["std_smooth"] * \
    test_df["bearing"].map(bearing_total) * 10
test_df["Lower_RUL_s"]     = np.maximum(
    test_df["Predicted_RUL_s"] - test_df["Uncertainty_s"], 0)
test_df["Upper_RUL_s"]     = test_df["Predicted_RUL_s"] + test_df["Uncertainty_s"]

test_df["Health_State"] = test_df["pred_smooth"].apply(classify_health)
print("Predictions done!")

Step 8: Final Results vs Actual RUL
Compare predictions against ground truth from PHM 2012 challenge PDF.

In [ ]:
actual_rul = {
    "Bearing1_3": 5730, "Bearing1_4": 339,  "Bearing1_5": 1610,
    "Bearing1_6": 1460, "Bearing1_7": 7570, "Bearing2_3": 7530,
    "Bearing2_4": 1390, "Bearing2_5": 3090, "Bearing2_6": 1290,
    "Bearing2_7": 580,  "Bearing3_3": 820
}

latest = test_df.sort_values("time_step").groupby("bearing").last().reset_index()

print(f"{'Bearing':<12} {'Predicted(s)':>13} {'Actual(s)':>10} {'Error%':>8} {'Uncertainty':>13} {'Health'}")
print("-" * 80)
errors = []
for _, row in latest.iterrows():
    b, pred_s = row["bearing"], row["Predicted_RUL_s"]
    act_s, unc_s = actual_rul[b], row["Uncertainty_s"]
    err = abs(pred_s - act_s) / act_s * 100
    errors.append(err)
    print(f"{b:<12} {pred_s:>13.0f} {act_s:>10} {err:>7.1f}% {unc_s:>12.0f}s  {row['Health_State']}")

print("-" * 80)
print(f"Average Error: {np.mean(errors):.1f}%")

 Step 9: Save Results

In [ ]:
test_df.to_csv("arya_test_predictions_timeseries.csv", index=False)
latest = test_df.sort_values("time_step").groupby("bearing").last().reset_index()
latest.to_csv("arya_test_predictions_final.csv", index=False)
print("Saved!")